### RAG PIPELINE  - Data Ingestion to Vector DB Pipeline


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


#### Read all PDF inside the directoy 

In [ ]:
def processAllPDF(pdf_directory):
    print(f"Process all PDFs inside {pdf_directory}")
    allProcessPDF = []
    try:
        pdfFiles = list(Path(pdf_directory).glob("**/*.pdf"))
        print(f"Number of PDF Files - {len(pdfFiles)}")
        for pdf in pdfFiles:
            
            document = PyMuPDFLoader(pdf)
            documentload = document.load()
            print(f"\nProcessing PDF : {pdf.name}")
            print(f"No of Pages : {len(documentload)} \n")
            
            for doc in documentload:
                doc.metadata["fileName"] = pdf.name
                doc.metadata["fileType"] = "pdf"
                allProcessPDF.append(doc)

        return allProcessPDF
    except Exception as e:  
        print(e)
        return []
    
all_processed_documencts = processAllPDF("../data")

all_processed_documencts

In [ ]:
def split_all_documents(documents,chunkSize=500,chunkOverlap=80):
    print(f"Length of Documents : {len(documents)}")
    doc_spilliter = RecursiveCharacterTextSplitter(
       chunk_size = chunkSize,
       chunk_overlap = chunkOverlap,
       length_function = len
    )

    splitdocs = doc_spilliter.split_documents(documents)

    print(f"Total Split Docs : {len(splitdocs)}")

    if splitdocs:
        print(f"Chunk Example Data")
        print(f"Metadata: {splitdocs[0].metadata}")
        print(f"Page Content : {splitdocs[0].page_content}")

    return splitdocs


splitdocs = split_all_documents(all_processed_documencts)
splitdocs

#### Text Spilliting get into chunks

### Embedding and Vector DB


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer #Embedding model will be available inside SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self,modal_name = "all-MiniLM-L6-v2"):
        self.modal_name = modal_name
        self.modal = None
        self._load_modal()

    def _load_modal(self):
        try:
            self.modal = SentenceTransformer(self.modal_name)
            print(f"Modal Loading : {self.modal_name} ")
            print(f"Modal Loaded Successfully.")
            print(f"Modal Embedding Dimesion : {self.modal.get_embedding_dimension()}")
        except Exception as e:
            print(e)

    def generate_embedding(self,texts : List[str]) -> np.ndarray:
        try:
            if not self.modal:
                raise ValueError("Modal not Loaded.")
            print(f"Generate embeddings for {len(texts)}")
            embedding = self.modal.encode(texts,show_progress_bar=True)
            print(f"Embedding Shape {embedding.shape}")
            return embedding
        except Exception as e:
            print(e)
            return []


embedding_manager = EmbeddingManager()
embedding = embedding_manager.generate_embedding([doc.page_content for doc in splitdocs])
embedding
#for page_content in splitdocs:


In [ ]:
class VectorStore:

    def __init__(self,collection_name:str = "pdf_documents", persisit_directory :str = "../data/vector"):
        """
        collection_name = Name of the ChromaDB collection
        persisit_directory = Directory to persist(store) the vector store
        """
        self.collection_name = collection_name
        self.persisit_directory = persisit_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize chromaDB collection and client"""
        try:
            os.makedirs(name=self.persisit_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persisit_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description":"PDF document embidding for RAG"})
            print(f"Vector Store Initialized . Collection : {self.collection_name}")
            print(f" {self.collection.count()}")
        
        except Exception as e:
            print(e)

    def add_documents(self,documents : List[Any] , embeddings : np.ndarray):
        """
        Add documents and their embeddings in vestor store
        """
        if len(documents) != len(embeddings):
            print("Number of documents and embeddings must be same")
            return

        idlist=[]
        metadatalist = []
        documetlist = []
        embeddinglist = []
        try:
            for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
                docid = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                idlist.append(docid)

                metadata = doc.metadata
                metadata["doc_index"] = i
                metadata["contentlength"] = len(doc.page_content)
                metadatalist.append(metadata)

                documetlist.append(doc.page_content)
                embeddinglist.append(embedding.tolist())

            
            self.collection.add(
                ids=idlist,
                metadatas= metadatalist,
                documents= documetlist,
                embeddings=embeddinglist
            )

            print(f"Successfully added {len(documents)} documents in vestore store")
            print(f"Total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(e)



vectorstore = VectorStore()
vectorstore.add_documents(splitdocs,embedding)
    

### Retriever PipeLine from Vector Store


In [ ]:
class RagRetriever:

    def __init__(self,vectorstore:VectorStore,embeddingManager:EmbeddingManager):
        self.vectorstore = vectorstore
        self.embeddingManager = embeddingManager

    def retrieve(self,query:str,top_k:int=5,threshold:float = 0.0) -> List[Dict[str,Any]]:
        print(query)

        queryEmbedding = self.embeddingManager.generate_embedding([query])[0]       

        results = self.vectorstore.collection.query(
            query_embeddings=queryEmbedding,
            n_results=top_k
        )
        retrieved_docs=[]
        print(results)
        if results["documents"] and results["documents"][0]:
            documents = results["documents"][0]
            print(documents)
            metadatas = results["metadatas"][0]
            ids = results["ids"][0]
            distances = results["distances"][0]

            for i,(document,metadata,id,distance) in enumerate(zip(documents,metadatas,ids,distances)):
                similarity_score = 1 - distance
                print(f"similarity_score : {similarity_score}")
                if similarity_score >= threshold :
                    retrieved_docs.append({
                        'id' : id,
                        'content' : document,
                        'metadata' : metadata,
                        'similarity_score' : similarity_score,
                        'distance' : distance,
                        'rank' : i+1
                    })
        
        return retrieved_docs

retriever = RagRetriever(vectorstore,embedding_manager)
#results = retriever.retrieve(query="Guidelines for goal settings")
#results

### Integration Vector DB Context Pipeline With LLM Output



In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

OpenAPIKEY = os.getenv("OPENAI_API_KEY")
modal = "gpt-4o-mini"

llm = ChatOpenAI(
    model=modal,
    api_key=OpenAPIKEY
)

def rag_simple(query,llm:ChatOpenAI,retirever:RagRetriever):

    results = retirever.retrieve(query)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    message = [
       {"role":"system","content":"You are an assistant, answer only from the supplied contecxt"},
       {"role":"user","content":f"""
         context:
         {context}
         Question:
         {query}       
       """}
    ]

    result = llm.invoke(message)
    return result
    


In [ ]:
result = rag_simple("Maternity Leave Ploicy",llm, retriever)
print(result.content)